# Generate Base and Fine-Tuned Evaluation Dataframes

This notebook uses `master_evaluation_manifest.jsonl` as the single source for both models and writes:

- `base_dataframe.jsonl`
- `finetuned_dataframe.jsonl`

Every original manifest field is preserved. The same 6,000 `evaluation_id` values remain in the same order in both outputs. Inference is deterministic and resumable; failed rows can be retried without repeating completed rows.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import sys
import time
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm

from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
PROJECT_DIR = Path(r"C:\Users\becker\Documents\Thesis")
MASTER_MANIFEST_PATH = PROJECT_DIR / "dataset"/ "manifests" / "master_evaluation_manifest.jsonl"

BASE_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
FINETUNED_ADAPTER_PATH = (
    PROJECT_DIR / "qwen3vl_bdd100k_lora" / "final_model"
)

IMAGE_SEARCH_ROOTS = [
    PROJECT_DIR,
    PROJECT_DIR / "dataset",
    PROJECT_DIR / "dataset" / "images",
    PROJECT_DIR / "dataset" / "images" / "val",
]

OUTPUT_DIR = PROJECT_DIR / "evaluation_outputs_master_6000"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

BASE_PREDICTION_LOG = CHECKPOINT_DIR / "base_predictions.jsonl"
FINETUNED_PREDICTION_LOG = CHECKPOINT_DIR / "finetuned_predictions.jsonl"

BASE_DATAFRAME_PATH = OUTPUT_DIR / "base_dataframe.jsonl"
FINETUNED_DATAFRAME_PATH = OUTPUT_DIR / "finetuned_dataframe.jsonl"
BASE_PICKLE_PATH = OUTPUT_DIR / "base_dataframe.pkl"
FINETUNED_PICKLE_PATH = OUTPUT_DIR / "finetuned_dataframe.pkl"
GENERATION_AUDIT_PATH = OUTPUT_DIR / "dataframe_generation_audit.json"

MIN_PIXELS = 256 * 256
MAX_PIXELS = 384 * 384

MAX_NEW_TOKENS_BY_TASK = {
    "Multilabel Object Recognition": 96,
    "Environmental Scene Understanding": 64,
    "Hazard and Anomaly Detection": 128,
    "Traffic Scene Captioning": 160,
}
DEFAULT_MAX_NEW_TOKENS = 128
MAX_INFERENCE_ATTEMPTS = 2

RUN_BASE_MODEL = True
RUN_FINETUNED_MODEL = True

RESET_BASE_CHECKPOINT = False
RESET_FINETUNED_CHECKPOINT = False
STRICT_FINAL_VALIDATION = True

# Keep as None for the full 6,000 rows.
ROW_LIMIT = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())

if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The configured BF16 inference requires a BF16-capable GPU.")

torch.set_float32_matmul_precision("high")

Python: c:\Users\becker\Documents\Thesis\venv\Scripts\python.exe
PyTorch: 2.5.1+cu121
Transformers: 4.57.0
CUDA available: True
GPU: NVIDIA RTX 4500 Ada Generation
CUDA runtime: 12.1
BF16 supported: True


## Manifest and JSONL utilities

In [5]:
def load_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"{Path(path).name}: invalid JSON on line {line_number}: {error}"
                ) from error
            if not isinstance(record, dict):
                raise TypeError(
                    f"{Path(path).name}: line {line_number} is not a JSON object."
                )
            records.append(record)
    return records


def write_jsonl(records, path):
    with Path(path).open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n"
            )


def append_jsonl(record, path):
    with Path(path).open("a", encoding="utf-8") as file:
        file.write(
            json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n"
        )


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [6]:
if not MASTER_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Master manifest not found: {MASTER_MANIFEST_PATH}")

master_records = load_jsonl(MASTER_MANIFEST_PATH)

if ROW_LIMIT is not None:
    master_records = master_records[:ROW_LIMIT]

required_fields = {
    "evaluation_id",
    "sample_id",
    "image_id",
    "image_path",
    "task",
    "instruction",
    "ground_truth_text",
    "master_row_index",
}

all_fields = set().union(*(record.keys() for record in master_records))
missing_fields = required_fields - all_fields

if missing_fields:
    raise ValueError(f"Missing master-manifest fields: {sorted(missing_fields)}")

for row_number, record in enumerate(master_records, start=1):
    for field in required_fields:
        if record.get(field) is None:
            raise ValueError(f"Row {row_number} has no value for {field!r}.")
    if not str(record["instruction"]).strip():
        raise ValueError(f"Row {row_number} has an empty instruction.")

evaluation_ids = [record["evaluation_id"] for record in master_records]

if len(evaluation_ids) != len(set(evaluation_ids)):
    raise ValueError("Duplicate evaluation_id values were found.")

task_counts = Counter(record["task"] for record in master_records)
manifest_sha256 = sha256_file(MASTER_MANIFEST_PATH)

print("Rows loaded:", len(master_records))
print("Unique evaluation IDs:", len(set(evaluation_ids)))
print("Manifest SHA256:", manifest_sha256)
print("\nRows per task:")
for task, count in task_counts.items():
    print(f"- {task}: {count}")

if ROW_LIMIT is None:
    expected_task_counts = {
        "Multilabel Object Recognition": 1500,
        "Environmental Scene Understanding": 1500,
        "Hazard and Anomaly Detection": 1500,
        "Traffic Scene Captioning": 1500,
    }
    if task_counts != expected_task_counts:
        raise ValueError(
            f"Unexpected full-manifest task distribution: {dict(task_counts)}"
        )

Rows loaded: 6000
Unique evaluation IDs: 6000
Manifest SHA256: 0ab22d29cdd73e0ce4e42dc98f3753bbed817c1a3afadad57de6a55352a2b314

Rows per task:
- Hazard and Anomaly Detection: 1500
- Traffic Scene Captioning: 1500
- Multilabel Object Recognition: 1500
- Environmental Scene Understanding: 1500


## Resolve and validate all image paths

In [7]:
def resolve_image_path(image_path):
    original_path = Path(str(image_path))
    candidates = []

    if original_path.is_absolute():
        candidates.append(original_path)

    for root in IMAGE_SEARCH_ROOTS:
        candidates.append(root / original_path)
        candidates.append(root / original_path.name)

    seen = set()

    for candidate in candidates:
        normalized = candidate.resolve()

        if normalized in seen:
            continue

        seen.add(normalized)

        if normalized.exists():
            return normalized

    return None


image_path_cache = {}
missing_images = []

for record in tqdm(master_records, desc="Resolving image paths"):
    manifest_image_path = record["image_path"]

    if manifest_image_path not in image_path_cache:
        image_path_cache[manifest_image_path] = resolve_image_path(
            manifest_image_path
        )

    resolved_path = image_path_cache[manifest_image_path]

    if resolved_path is None:
        missing_images.append(
            {
                "evaluation_id": record["evaluation_id"],
                "image_path": manifest_image_path,
            }
        )
    else:
        record["resolved_image_path"] = str(resolved_path)

print("Rows:", len(master_records))
print("Unique manifest image paths:", len(image_path_cache))
print("Missing image rows:", len(missing_images))

if missing_images:
    display(pd.DataFrame(missing_images).head(20))
    raise FileNotFoundError(
        "Some images could not be found. Correct IMAGE_SEARCH_ROOTS first."
    )

Resolving image paths:   0%|          | 0/6000 [00:00<?, ?it/s]

Rows: 6000
Unique manifest image paths: 4643
Missing image rows: 0


## Validate the LoRA adapter

In [8]:
if not FINETUNED_ADAPTER_PATH.exists():
    raise FileNotFoundError(
        f"Fine-tuned adapter directory not found: {FINETUNED_ADAPTER_PATH}"
    )

adapter_config_path = FINETUNED_ADAPTER_PATH / "adapter_config.json"

if not adapter_config_path.exists():
    raise FileNotFoundError(
        f"adapter_config.json not found inside {FINETUNED_ADAPTER_PATH}"
    )

adapter_config = json.loads(
    adapter_config_path.read_text(encoding="utf-8")
)

print("Adapter path:", FINETUNED_ADAPTER_PATH)
print("Adapter base model:", adapter_config.get("base_model_name_or_path"))
print("Adapter type:", adapter_config.get("peft_type"))

configured_adapter_base = adapter_config.get("base_model_name_or_path")

if configured_adapter_base and configured_adapter_base != BASE_MODEL_ID:
    print(
        "Warning: adapter_config.json references "
        f"{configured_adapter_base!r}, while BASE_MODEL_ID is {BASE_MODEL_ID!r}."
    )

Adapter path: C:\Users\becker\Documents\Thesis\qwen3vl_bdd100k_lora\final_model
Adapter base model: Qwen/Qwen3-VL-4B-Instruct
Adapter type: LORA


## Resumable checkpoints

In [9]:
def select_best_prediction_records(path):
    path = Path(path)

    if not path.exists():
        return {}

    best = {}

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            record = json.loads(line)

            if record.get("manifest_sha256") != manifest_sha256:
                raise ValueError(
                    f"{path.name}: checkpoint line {line_number} was generated "
                    "from a different master manifest."
                )

            evaluation_id = record["evaluation_id"]
            previous = best.get(evaluation_id)

            if previous is None:
                best[evaluation_id] = record
                continue

            previous_completed = previous.get("eval_status") == "Completed"
            current_completed = record.get("eval_status") == "Completed"

            if current_completed or not previous_completed:
                best[evaluation_id] = record

    return best


def completed_evaluation_ids(path):
    return {
        evaluation_id
        for evaluation_id, record in select_best_prediction_records(path).items()
        if record.get("eval_status") == "Completed"
    }


def reset_checkpoint(path, should_reset):
    path = Path(path)
    if should_reset and path.exists():
        path.unlink()
        print("Deleted checkpoint:", path)


reset_checkpoint(BASE_PREDICTION_LOG, RESET_BASE_CHECKPOINT)
reset_checkpoint(FINETUNED_PREDICTION_LOG, RESET_FINETUNED_CHECKPOINT)

## Model loading and deterministic inference

In [9]:
def load_base_processor():
    return AutoProcessor.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
    )


def load_base_model():
    return (
        AutoModelForImageTextToText.from_pretrained(
            BASE_MODEL_ID,
            torch_dtype=torch.bfloat16,
            attn_implementation="sdpa",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        .to("cuda")
        .eval()
    )


def load_finetuned_processor():
    try:
        processor = AutoProcessor.from_pretrained(
            FINETUNED_ADAPTER_PATH,
            trust_remote_code=True,
        )
        print("Loaded processor from the fine-tuned directory.")
        return processor
    except Exception as error:
        print(
            "Could not load a processor from the fine-tuned directory. "
            f"Falling back to {BASE_MODEL_ID}. Reason: {error}"
        )
        return load_base_processor()


def load_finetuned_model():
    base_model = load_base_model()
    model = PeftModel.from_pretrained(
        base_model,
        FINETUNED_ADAPTER_PATH,
        is_trainable=False,
    )
    model.eval()
    return model


def get_model_device(model):
    return next(model.parameters()).device


def generate_prediction(record, model, processor):
    image_path = record["resolved_image_path"]
    instruction = record["instruction"]
    task = record["task"]

    max_new_tokens = MAX_NEW_TOKENS_BY_TASK.get(
        task,
        DEFAULT_MAX_NEW_TOKENS,
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path,
                    "min_pixels": MIN_PIXELS,
                    "max_pixels": MAX_PIXELS,
                },
                {
                    "type": "text",
                    "text": instruction,
                },
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    device = get_model_device(model)

    inputs = {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in inputs.items()
    }

    input_token_count = int(inputs["input_ids"].shape[-1])

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    torch.cuda.synchronize()
    latency_seconds = time.perf_counter() - start_time

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs["input_ids"],
            generated_ids,
        )
    ]

    output_token_count = int(generated_ids_trimmed[0].numel())

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return {
        "predicted_output": output_text,
        "input_token_count": input_token_count,
        "output_token_count": output_token_count,
        "max_new_tokens": max_new_tokens,
        "latency_seconds": latency_seconds,
    }


def run_model_inference(
    records,
    model,
    processor,
    model_variant,
    model_reference,
    prediction_log_path,
):
    completed_ids = completed_evaluation_ids(prediction_log_path)

    print(
        f"{model_variant}: {len(completed_ids)} completed rows "
        "already present."
    )

    for record in tqdm(
        records,
        total=len(records),
        desc=model_variant,
    ):
        evaluation_id = record["evaluation_id"]

        if evaluation_id in completed_ids:
            continue

        prediction = None
        final_error = None
        attempt_count = 0

        for attempt in range(1, MAX_INFERENCE_ATTEMPTS + 1):
            attempt_count = attempt

            try:
                prediction = generate_prediction(
                    record=record,
                    model=model,
                    processor=processor,
                )
                final_error = None
                break
            except Exception as error:
                final_error = error
                gc.collect()
                torch.cuda.empty_cache()

        output_record = {
            "evaluation_id": evaluation_id,
            "master_row_index": int(record["master_row_index"]),
            "task": record["task"],
            "model_variant": model_variant,
            "model_reference": str(model_reference),
            "manifest_sha256": manifest_sha256,
            "attempt_count": attempt_count,
            "inference_timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }

        if prediction is not None:
            output_record.update(prediction)
            output_record["eval_status"] = "Completed"
            output_record["generation_error"] = None
        else:
            output_record.update(
                {
                    "predicted_output": None,
                    "input_token_count": None,
                    "output_token_count": None,
                    "max_new_tokens": MAX_NEW_TOKENS_BY_TASK.get(
                        record["task"],
                        DEFAULT_MAX_NEW_TOKENS,
                    ),
                    "latency_seconds": None,
                    "eval_status": "Failed",
                    "generation_error": (
                        f"{type(final_error).__name__}: {final_error}"
                    ),
                }
            )

        append_jsonl(output_record, prediction_log_path)

    print(f"{model_variant} inference pass finished.")

## Run the base model

In [10]:
if RUN_BASE_MODEL:
    base_processor = load_base_processor()
    base_model = load_base_model()

    print("Base model device:", get_model_device(base_model))

    run_model_inference(
        records=master_records,
        model=base_model,
        processor=base_processor,
        model_variant="base",
        model_reference=BASE_MODEL_ID,
        prediction_log_path=BASE_PREDICTION_LOG,
    )

    del base_model
    del base_processor

    gc.collect()
    torch.cuda.empty_cache()

    print(
        "CUDA memory after unloading:",
        round(torch.cuda.memory_allocated() / 1024**3, 3),
        "GB",
    )
else:
    print("Base-model inference is disabled.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model device: cuda:0
base: 0 completed rows already present.


base:   0%|          | 0/6000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


base inference pass finished.
CUDA memory after unloading: 0.008 GB


## Run the fine-tuned LoRA model

In [11]:
if RUN_FINETUNED_MODEL:
    finetuned_processor = load_finetuned_processor()
    finetuned_model = load_finetuned_model()

    print("Fine-tuned model device:", get_model_device(finetuned_model))

    run_model_inference(
        records=master_records,
        model=finetuned_model,
        processor=finetuned_processor,
        model_variant="finetuned",
        model_reference=FINETUNED_ADAPTER_PATH,
        prediction_log_path=FINETUNED_PREDICTION_LOG,
    )

    del finetuned_model
    del finetuned_processor

    gc.collect()
    torch.cuda.empty_cache()

    print(
        "CUDA memory after unloading:",
        round(torch.cuda.memory_allocated() / 1024**3, 3),
        "GB",
    )
else:
    print("Fine-tuned-model inference is disabled.")

Loaded processor from the fine-tuned directory.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Fine-tuned model device: cuda:0
finetuned: 0 completed rows already present.


finetuned:   0%|          | 0/6000 [00:00<?, ?it/s]

finetuned inference pass finished.
CUDA memory after unloading: 0.008 GB


## Construct the final aligned dataframes

Failed rows are retained so later evaluation can measure inference failures and invalid outputs.

Compatibility aliases are added:

- `ground_truth = ground_truth_text`
- `task_category = task`

In [10]:
PREDICTION_FIELDS = [
    "predicted_output",
    "eval_status",
    "generation_error",
    "model_variant",
    "model_reference",
    "attempt_count",
    "input_token_count",
    "output_token_count",
    "max_new_tokens",
    "latency_seconds",
    "inference_timestamp_utc",
    "manifest_sha256",
]


def build_final_dataframe_records(
    manifest_records,
    prediction_log_path,
    model_variant,
):
    prediction_map = select_best_prediction_records(prediction_log_path)
    final_records = []

    for manifest_record in manifest_records:
        evaluation_id = manifest_record["evaluation_id"]
        output_record = dict(manifest_record)
        prediction = prediction_map.get(evaluation_id)

        if prediction is None:
            for field in PREDICTION_FIELDS:
                output_record[field] = None
            output_record["model_variant"] = model_variant
            output_record["eval_status"] = "Missing prediction record"
        else:
            for field in PREDICTION_FIELDS:
                output_record[field] = prediction.get(field)

        output_record["ground_truth"] = output_record["ground_truth_text"]
        output_record["task_category"] = output_record["task"]
        final_records.append(output_record)

    return final_records


base_dataframe_records = build_final_dataframe_records(
    master_records,
    BASE_PREDICTION_LOG,
    "base",
)

finetuned_dataframe_records = build_final_dataframe_records(
    master_records,
    FINETUNED_PREDICTION_LOG,
    "finetuned",
)

write_jsonl(base_dataframe_records, BASE_DATAFRAME_PATH)
write_jsonl(finetuned_dataframe_records, FINETUNED_DATAFRAME_PATH)

base_dataframe = pd.DataFrame(base_dataframe_records)
finetuned_dataframe = pd.DataFrame(finetuned_dataframe_records)

base_dataframe.to_pickle(BASE_PICKLE_PATH)
finetuned_dataframe.to_pickle(FINETUNED_PICKLE_PATH)

print("Saved base dataframe:", BASE_DATAFRAME_PATH)
print("Saved fine-tuned dataframe:", FINETUNED_DATAFRAME_PATH)
print("Base shape:", base_dataframe.shape)
print("Fine-tuned shape:", finetuned_dataframe.shape)

Saved base dataframe: C:\Users\becker\Documents\Thesis\evaluation_outputs_master_6000\base_dataframe.jsonl
Saved fine-tuned dataframe: C:\Users\becker\Documents\Thesis\evaluation_outputs_master_6000\finetuned_dataframe.jsonl
Base shape: (6000, 73)
Fine-tuned shape: (6000, 73)


## Final alignment and completion audit

In [11]:
def dataframe_audit(dataframe, name):
    return {
        "name": name,
        "rows": int(len(dataframe)),
        "unique_evaluation_ids": int(
            dataframe["evaluation_id"].nunique()
        ),
        "task_counts": {
            str(key): int(value)
            for key, value in dataframe[
                "task"
            ].value_counts().to_dict().items()
        },
        "status_counts": {
            str(key): int(value)
            for key, value in dataframe[
                "eval_status"
            ].value_counts(dropna=False).to_dict().items()
        },
        "missing_predictions": int(
            dataframe["predicted_output"].isna().sum()
        ),
        "empty_predictions": int(
            dataframe["predicted_output"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),
        "mean_latency_seconds": (
            float(dataframe["latency_seconds"].dropna().mean())
            if dataframe["latency_seconds"].notna().any()
            else None
        ),
        "mean_output_tokens": (
            float(dataframe["output_token_count"].dropna().mean())
            if dataframe["output_token_count"].notna().any()
            else None
        ),
    }


expected_order = [record["evaluation_id"] for record in master_records]
base_order = list(base_dataframe["evaluation_id"])
finetuned_order = list(finetuned_dataframe["evaluation_id"])

if base_order != expected_order:
    raise RuntimeError(
        "Base dataframe order is not aligned with the master manifest."
    )

if finetuned_order != expected_order:
    raise RuntimeError(
        "Fine-tuned dataframe order is not aligned with the master manifest."
    )

if base_order != finetuned_order:
    raise RuntimeError(
        "Base and fine-tuned dataframes are not row-aligned."
    )

base_audit = dataframe_audit(base_dataframe, "base")
finetuned_audit = dataframe_audit(finetuned_dataframe, "finetuned")

generation_audit = {
    "manifest_path": str(MASTER_MANIFEST_PATH),
    "manifest_sha256": manifest_sha256,
    "master_rows": len(master_records),
    "base_model_id": BASE_MODEL_ID,
    "finetuned_adapter_path": str(FINETUNED_ADAPTER_PATH),
    "pixel_limits": {
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
    },
    "max_new_tokens_by_task": MAX_NEW_TOKENS_BY_TASK,
    "base_dataframe": base_audit,
    "finetuned_dataframe": finetuned_audit,
    "alignment": {
        "base_matches_master_order": True,
        "finetuned_matches_master_order": True,
        "base_matches_finetuned_order": True,
    },
}

GENERATION_AUDIT_PATH.write_text(
    json.dumps(
        generation_audit,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print(json.dumps(generation_audit, indent=2, ensure_ascii=False))
print("\nAudit saved:", GENERATION_AUDIT_PATH)

if STRICT_FINAL_VALIDATION:
    incomplete_base = base_dataframe["eval_status"].ne("Completed").sum()
    incomplete_finetuned = (
        finetuned_dataframe["eval_status"].ne("Completed").sum()
    )

    if incomplete_base or incomplete_finetuned:
        raise RuntimeError(
            "The dataframe files were saved, but some rows are incomplete. "
            f"Base incomplete: {incomplete_base}; "
            f"fine-tuned incomplete: {incomplete_finetuned}. "
            "Rerun the model cells. Completed rows will be skipped and "
            "failed rows retried."
        )

print("\nBoth aligned dataframes are complete.")

{
  "manifest_path": "C:\\Users\\becker\\Documents\\Thesis\\dataset\\manifests\\master_evaluation_manifest.jsonl",
  "manifest_sha256": "0ab22d29cdd73e0ce4e42dc98f3753bbed817c1a3afadad57de6a55352a2b314",
  "master_rows": 6000,
  "base_model_id": "Qwen/Qwen3-VL-4B-Instruct",
  "finetuned_adapter_path": "C:\\Users\\becker\\Documents\\Thesis\\qwen3vl_bdd100k_lora\\final_model",
  "pixel_limits": {
    "min_pixels": 65536,
    "max_pixels": 147456
  },
  "max_new_tokens_by_task": {
    "Multilabel Object Recognition": 96,
    "Environmental Scene Understanding": 64,
    "Hazard and Anomaly Detection": 128,
    "Traffic Scene Captioning": 160
  },
  "base_dataframe": {
    "name": "base",
    "rows": 6000,
    "unique_evaluation_ids": 6000,
    "task_counts": {
      "Hazard and Anomaly Detection": 1500,
      "Traffic Scene Captioning": 1500,
      "Multilabel Object Recognition": 1500,
      "Environmental Scene Understanding": 1500
    },
    "status_counts": {
      "Completed": 6000
  

In [12]:
display_columns = [
    "master_row_index",
    "evaluation_id",
    "task",
    "image_path",
    "instruction",
    "ground_truth",
    "predicted_output",
    "eval_status",
    "output_token_count",
    "latency_seconds",
]

display(base_dataframe[display_columns].head(8))
display(finetuned_dataframe[display_columns].head(8))

,master_row_index,evaluation_id,task,image_path,instruction,ground_truth,predicted_output,eval_status,output_token_count,latency_seconds
0,1,hazard_and_anomaly_detection::had_pos_0538,Hazard and Anomaly Detection,val/images/c7415d8d-dfb8ef47.jpg,Determine whether the image contains a rule-de...,A visible person is inside the directly drivab...,Decision: No\n\nHazard Type: None\n\nInvolved ...,Completed,128,7.306360
1,2,traffic_scene_captioning::sc_rep_0958,Traffic Scene Captioning,val/images/c0905216-425f59d3.jpg,Describe the traffic scene in one concise para...,A clear nighttime city street scene featuring ...,"The scene is a dark, nighttime road with low v...",Completed,127,6.055853
2,3,traffic_scene_captioning::sc_rep_0679,Traffic Scene Captioning,val/images/c9d7b898-27eba05f.jpg,Describe the traffic scene in one concise para...,A clear daytime city street scene featuring a ...,"The scene depicts a clear, sunny day with brig...",Completed,148,6.564958
3,4,traffic_scene_captioning::sc_chal_0238,Traffic Scene Captioning,val/images/bb6575eb-90478df6.jpg,Describe the traffic scene in one concise para...,A rainy highway scene at dawn or dusk featurin...,The scene is a multi-lane highway during twili...,Completed,107,5.805808
4,5,multilabel_object_recognition::mor_rep_0438,Multilabel Object Recognition,val/images/bd989210-afaf7243.jpg,Identify all visible traffic-object classes in...,car,"car, traffic sign, traffic light",Completed,8,0.556510
5,6,traffic_scene_captioning::sc_rep_0094,Traffic Scene Captioning,val/images/c13ab9cd-88c99ae5.jpg,Describe the traffic scene in one concise para...,A rainy nighttime city street scene featuring ...,"At night, a wet urban street is filled with tr...",Completed,125,6.011982
6,7,traffic_scene_captioning::sc_chal_0072,Traffic Scene Captioning,val/images/b5ab0e46-8eab4733.jpg,Describe the traffic scene in one concise para...,A clear daytime city street scene featuring a ...,The scene depicts a sunny daytime urban street...,Completed,122,5.702233
7,8,hazard_and_anomaly_detection::had_pos_0478,Hazard and Anomaly Detection,val/images/b98f4bb1-100931bb.jpg,Determine whether the image contains a rule-de...,A visible person is inside the directly drivab...,Decision: Yes\n\nHazard Type: Sufficiently lar...,Completed,128,6.220018


,master_row_index,evaluation_id,task,image_path,instruction,ground_truth,predicted_output,eval_status,output_token_count,latency_seconds
0,1,hazard_and_anomaly_detection::had_pos_0538,Hazard and Anomaly Detection,val/images/c7415d8d-dfb8ef47.jpg,Determine whether the image contains a rule-de...,A visible person is inside the directly drivab...,Potential hazard: 10 pedestrian(s)/rider(s) ne...,Completed,48,3.691741
1,2,traffic_scene_captioning::sc_rep_0958,Traffic Scene Captioning,val/images/c0905216-425f59d3.jpg,Describe the traffic scene in one concise para...,A clear nighttime city street scene featuring ...,This is a driving scene captured during nightt...,Completed,27,2.080292
2,3,traffic_scene_captioning::sc_rep_0679,Traffic Scene Captioning,val/images/c9d7b898-27eba05f.jpg,Describe the traffic scene in one concise para...,A clear daytime city street scene featuring a ...,This is a driving scene captured during daytim...,Completed,29,2.704203
3,4,traffic_scene_captioning::sc_chal_0238,Traffic Scene Captioning,val/images/bb6575eb-90478df6.jpg,Describe the traffic scene in one concise para...,A rainy highway scene at dawn or dusk featurin...,This is a highway scene captured during dawn o...,Completed,32,2.374197
4,5,multilabel_object_recognition::mor_rep_0438,Multilabel Object Recognition,val/images/bd989210-afaf7243.jpg,Identify all visible traffic-object classes in...,car,"car, traffic sign, truck, bus, person, rider, ...",Completed,25,1.671950
5,6,traffic_scene_captioning::sc_rep_0094,Traffic Scene Captioning,val/images/c13ab9cd-88c99ae5.jpg,Describe the traffic scene in one concise para...,A rainy nighttime city street scene featuring ...,This is a nighttime city scene with 15 detecte...,Completed,50,3.472509
6,7,traffic_scene_captioning::sc_chal_0072,Traffic Scene Captioning,val/images/b5ab0e46-8eab4733.jpg,Describe the traffic scene in one concise para...,A clear daytime city street scene featuring a ...,This is a driving scene captured during daytim...,Completed,29,2.105176
7,8,hazard_and_anomaly_detection::had_pos_0478,Hazard and Anomaly Detection,val/images/b98f4bb1-100931bb.jpg,Determine whether the image contains a rule-de...,A visible person is inside the directly drivab...,Potential hazard: 1 pedestrian(s)/rider(s) nea...,Completed,47,3.386676
